# Mobility data audit and causal research design

**Chicago grocery study, 2021–2025**

This notebook records the status of the observed-data extension. It does not manufacture observations or regression results when the licensed input is absent.

Publication record: Published · 17 November 2026 · v1.0.0. [Website](https://emmanuelsampson-prog.github.io/grocery-access-chicago/) · [Repository](https://github.com/emmanuelsampson-prog/grocery-access-chicago). All maps and charts are generated with Matplotlib.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import geopandas as gpd
from IPython.display import display, Image, Markdown
pd.set_option('display.max_columns', 20)
DATA = ROOT / 'data/processed'
FIG = ROOT / 'outputs/figures'


## 1. Input availability
Penn institutional access to Weekly Patterns Plus is verified. The provider terms await the account holder’s review before download. The expected file is private and excluded from public packaging.

In [2]:
from src.mobility import audit_panel
status = audit_panel(ROOT/'data/restricted/store_week.csv')
display(pd.Series(status))

status                                awaiting_licensed_data
rows                                                       0
message    No observed store-week mobility panel has been...
dtype: object

## 2. Events and weekly alignment
Use the first full post-event week. Exclude the transition week from a weekly specification; test announcement timing separately. Whole Foods vacancy ends before the replacement opening, so it cannot supply a full year of post-closure vacancy observations.

In [3]:
from src.mobility import first_full_post_week
events = pd.read_csv(DATA/'verified_events.csv')
events['first_full_post_week'] = events.event_date.map(first_full_post_week)
display(events[['label','event_date','first_full_post_week']])

,label,event_date,first_full_post_week
0,"Whole Foods, Englewood",2022-11-13,2022-11-14
1,"Walmart, Chatham",2023-04-16,2023-04-17
2,"Walmart, Kenwood",2023-04-16,2023-04-17
3,"Walmart, Lakeview",2023-04-16,2023-04-17
4,"Walmart, Little Village",2023-04-16,2023-04-17
5,"Save A Lot, Englewood",2023-05-11,2023-05-15


## 3. Estimands and spatial spillovers
Estimate redistribution toward surviving destinations and origin-area shopping access. Closed-store visits mechanically fall; that alone is not a welfare or travel effect. Candidate comparison areas need stable pretrends and no overlapping closures or openings. Nearby survivors can receive diverted demand and should not automatically be controls. With few independent events, tract-level sample size does not solve weak treatment-level inference.

Use event-specific estimates and clean comparisons. Treat the Englewood reopening as a separate intervention, rather than fitting one permanent post-closure indicator. See `docs/research_protocol.md` for identification assumptions and references.

## 4. What remains unestimated
Observed destination substitution; actual journey distance; walking or driving shifts; realized emissions; causal event-study effects. The completed access model is useful descriptive evidence, but it cannot supply these missing outcomes.

In [4]:
display(Markdown((ROOT/'docs/mobility_data_request.md').read_text()))

# Mobility data specification

Study window: 1 January 2021 through 31 December 2025. Publication details: see publication.json.

Penn Libraries documents institutional access to Dewey at https://www.library.upenn.edu/lippincott/datapoints/data-access-dewey-2025. Institutional access to Weekly Patterns Plus was verified on 16 November 2026 (America/New_York). The catalog lists coverage from 2 January 2017 to 19 October 2026 and a 7 November 2026 refresh. The provider's additional terms still require the account holder's review and acceptance before download. No mobility export has been downloaded for this project.

Dataset citation: Advan Research (2025), Foot Traffic / Weekly Patterns Plus, Dewey Data, https://doi.org/10.82551/C103-N851. This records availability, not completed use of the licensed data. Do not redistribute the source table or private account information in the project.

## Request and geographic extent

Use a single consistent version of Advan Weekly Patterns or Weekly Patterns Plus for Chicago and a 10 km buffer. Request all grocery competitors in this extent, not only the six focal records. The verified event ledger provides matching addresses and company store numbers; a reviewer must resolve provider POI identities, geofences, relocations and changes of brand. Whole Foods and Save A Lot at 832 W 63rd Street require distinct business identities even when the physical geofence is shared.

Retain local Monday-start weeks; persistent store and location identifiers; brand; address; coordinates; NAICS; open and close metadata; visits and visitors; provider product/version; sampling/normalization fields and flags; and authorized aggregate home-origin fields when permitted. Never publish device traces or small origin cells. Public release of licensed aggregates is subject to the actual contract, not inferred from institutional access.

## Canonical local input

Save a reviewed aggregate file as `data/restricted/store_week.csv` with:

| Column | Meaning |
| --- | --- |
| store_id | Reviewed stable business identifier |
| week | Local Monday date, YYYY-MM-DD |
| visits | Provider visit count or estimate, with units documented |
| visitors | Provider unique visitor count or estimate |
| longitude, latitude | Reviewed POI coordinate, EPSG:4326 |
| provider_product, provider_version | Product and fixed release/version |
| normalization_denominator | Optional documented sampling denominator |
| suppressed | Optional provider suppression indicator |

The canonical schema intentionally does not silently map `raw_visit_counts` to `visit_counts`: different products have different meanings. Document the conversion, version and units before running the validator. Missing and suppressed rows must not be converted to zero. Retain actual provider file checksums in a private manifest.

## Interpretation checks

Advan's current documentation (https://docs.advanresearch.com/) distinguishes visitation and trade-area panels. Trade-area origin values should be treated as ratios or scaled indicators rather than absolute longitudinal visitor counts. Its distance-from-home field is a median haversine distance from a home geohash to the POI, not a measured street journey. Neither this field nor a foot-traffic count identifies walking or driving.

Audit at least a full seasonal cycle before and after each event where the event history permits it. For the Whole Foods vacancy effect, the post-closure window ends before Save A Lot opens; a full year of untreated vacancy is not available. Record announced closures and anticipation: Whole Foods announced its exit months before its last day, while Walmart announced the four closures on 11 April 2023. Examine both announcement and operating dates in sensitivity work.

## Required before estimating an effect

1. Match identities and geofences and inspect known business-date discrepancies.
2. Plot coverage, missingness, panel normalization and seasonality before fitting models.
3. Audit other grocery changes in treated and comparison catchments.
4. Define treatment, spatial spillovers and controls before inspecting treatment coefficients.
5. Review the license for permitted aggregate outputs; keep raw inputs private.
6. Report comparisons as descriptive if the data cannot support causal identification.
